In [ ]:
#work in progress.

import os
import re
import pandas as pd
from dotenv import load_dotenv

load_dotenv()


# Define Taxonomy Dictionary (Main Headings without numbering)
TAXONOMY = {
    "Metallic Minerals": {
        "Iron Ore": [
            "Iron Ore",
            "Hematite",
            "Magnetite",
            "Siderite",
            "Goethite",
            "Iron Sand",
        ],
        "Bauxite": [
            "Bauxite",
            "Aluminium Ore",
            "Alumina Ore",
            "Hydrous Aluminum Oxide",
        ],
        "Copper Ore": [
            "Copper Ore",
            "Copper",
            "Chalcopyrite",
            "Bornite",
            "Malachite",
            "Cuprite",
        ],
        "Manganese Ore": [
            "Manganese Ore",
            "Manganese",
            "Pyrolusite",
            "Psilomelane",
            "Rhodochrosite",
        ],
        "Chromite": ["Chromite", "Chromium Ore", "Ferrochrome Ore"],
        "Gold": [
            "Gold",
            "Native Gold",
            "Gold Ore",
            "Auriferous Quartz",
            "Placer Gold",
        ],
        "Silver": ["Silver", "Native Silver", "Argentite", "Galvanic Silver"],
        "Lead & Zinc": [
            "Lead & Zinc",
            "Lead",
            "Zinc",
            "Galena",
            "Sphalerite",
            "Zinc Blende",
            "Calamine",
        ],
        "Nickel": ["Nickel", "Lateritic Nickel", "Pentandite"],
        "Titanium Ores": [
            "Titanium Ores",
            "Titanium",
            "Ilmenite",
            "Rutile",
            "Titaniferous Magnetite",
        ],
        "Tin": ["Tin", "Cassiterite", "Tinstone"],
    },
    "Energy & Strategic / Atomic Minerals": {
        "Coal": [
            "Coal",
            "Thermal Coal",
            "Coking Coal",
            "Bituminous",
            "Lignite",
            "Brown Coal",
            "Anthracite",
        ],
        "Petroleum & Natural Gas": [
            "Petroleum & Natural Gas",
            "Petroleum",
            "Crude Oil",
            "Mineral Oil",
            "Hydrocarbons",
            "Natural Gas",
            "Shale Gas",
        ],
        "Uranium": ["Uranium", "Pitchblende", "Uraninite"],
        "Thorium": ["Thorium", "Monazite Sand", "Thorianite"],
        "Lithium": ["Lithium", "Spodumene", "Lepidolite", "White Gold"],
    },
    "Non-Metallic & Industrial Minerals": {
        "Limestone": [
            "Limestone",
            "Calcium Carbonate",
            "Calcite",
            "Chalk",
            "Quicklime Stone",
        ],
        "Dolomite": ["Dolomite", "Dolostone", "Magnesium Limestone"],
        "Mica": [
            "Mica",
            "Muscovite",
            "Phlogopite",
            "Biotite",
            "Isinglass",
            "Sheet Mica",
            "Mica Flakes",
        ],
        "Gypsum": ["Gypsum", "Selenite", "Alabaster", "Hydrated Calcium Sulfate"],
        "Rock Phosphate": ["Rock Phosphate", "Phosphorite", "Apatite", "Phosphate Rock"],
        "Fluorspar": ["Fluorspar", "Fluorite", "Calcium Fluoride"],
        "Barite": ["Barite", "Barytes", "Heavy Spar", "Barium Sulfate"],
        "Magnesite": ["Magnesite", "Magnesium Carbonate"],
        "Kyanite & Sillimanite": [
            "Kyanite & Sillimanite",
            "Kyanite",
            "Sillimanite",
            "Aluminosilicate Minerals",
            "Refractory Minerals",
        ],
        "Asbestos": ["Asbestos", "Chrysotile", "Amphibole", "White Asbestos"],
        "Feldspar": ["Feldspar", "Orthoclase", "Plagioclase", "Microcline"],
        "Quartz & Silica": [
            "Quartz & Silica",
            "Silica Sand",
            "Quartzite",
            "Crystal Quartz",
            "Rock Crystal",
        ],
        "Talc & Soapstone": [
            "Talc & Soapstone",
            "Talc",
            "Soapstone",
            "Steatite",
            "French Chalk",
            "Hydrous Magnesium Silicate",
        ],
    },
    "Gemstones & Precious Stones": {
        "Diamond": [
            "Diamond",
            "Carbon Crystal",
            "Gem Diamond",
            "Industrial Diamond",
            "Bort",
        ],
        "Emerald": ["Emerald", "Green Beryl"],
        "Ruby & Sapphire": [
            "Ruby & Sapphire",
            "Ruby",
            "Sapphire",
            "Corundum",
            "Manik",
            "Neelam",
        ],
        "Garnet": ["Garnet", "Almandine", "Pyrope", "Garnet Sand"],
        "Agate & Chalcedony": [
            "Agate & Chalcedony",
            "Agate",
            "Chalcedony",
            "Carnelian",
            "Jasper",
            "Onyx",
            "Silica Stones",
        ],
    },
    "Building & Dimension Stones": {
        "Granite": [
            "Granite",
            "Dimension Stone",
            "Commercial Granite",
            "Black Granite",
        ],
        "Marble": [
            "Marble",
            "Makrana Marble",
            "Calcitic Marble",
            "Dolomitic Marble",
            "Crystalline Limestone",
        ],
        "Sandstone": [
            "Sandstone",
            "Red Fort Stone",
            "Dholpur Stone",
            "Buff Sandstone",
            "Quartzose Sandstone",
        ],
        "Slate & Schist": [
            "Slate & Schist",
            "Slate",
            "Schist",
            "Roofing Slate",
            "Flagstone",
        ],
        "Laterite": ["Laterite", "Laterite Stone", "Building Block Stone"],
        "Basalt & Trap Rock": [
            "Basalt & Trap Rock",
            "Basalt",
            "Trap Rock",
            "Deccan Trap",
            "Black Trap",
            "Crushed Stone",
        ],
    },
    "Sand, Gravel & Aggregates": {
        "River Sand": ["River Sand", "Natural Sand", "Coarse Sand", "Bajri", "Reti"],
        "Silica Sand": ["Silica Sand", "Glass Sand", "Industrial Sand", "Quartz Sand"],
        "Manufactured Sand": [
            "Manufactured Sand",
            "M-Sand",
            "Crushed Rock Sand",
            "Artificial Sand",
        ],
        "Gravel & Aggregate": [
            "Gravel & Aggregate",
            "Gravel",
            "Aggregate",
            "Jit",
            "Gitti",
            "Metal Stones",
            "Crushed Stone Aggregate",
        ],
        "Placer & Beach Sands": [
            "Placer & Beach Sands",
            "Monazite Sand",
            "Heavy Mineral Sand",
            "Black Sand",
        ],
    },
}


def classify_text(text, taxonomy):
    """
    Scans project text for mineral keywords and extracts Main Heading, Subheading, and Keyword.
    If multiple minerals are found, joins them using comma separation.
    """
    if not isinstance(text, str) or not text.strip():
        return "Unclassified", "Unclassified", "None"

    # Flatten taxonomy keywords and sort by keyword length descending (longer phrases match first)
    keyword_map = []
    for main_heading, subheadings in taxonomy.items():
        for subheading, keywords in subheadings.items():
            all_kw = set([subheading] + keywords)
            for kw in all_kw:
                keyword_map.append((kw, main_heading, subheading))

    keyword_map.sort(key=lambda x: len(x[0]), reverse=True)

    matched_main = []
    matched_sub = []
    matched_kw = []
    seen_subheadings = set()

    for kw, main_h, sub_h in keyword_map:
        # Regex search using word boundaries (\b) and case insensitivity
        pattern = r"\b" + re.escape(kw) + r"\b"
        if re.search(pattern, text, flags=re.IGNORECASE):
            if (main_h, sub_h) not in seen_subheadings:
                seen_subheadings.add((main_h, sub_h))
                matched_main.append(main_h)
                matched_sub.append(sub_h)
                matched_kw.append(kw)

    if matched_main:
        # Deduplicate main headings while preserving order
        unique_mains = list(dict.fromkeys(matched_main))
        return (
            ", ".join(unique_mains),
            ", ".join(matched_sub),
            ", ".join(matched_kw),
        )
    else:
        return "Unclassified", "Unclassified", "None"


def update_excel_file(file_path, project_col_name="Project Name"):
    """
    Reads the Excel file, adds/updates classification columns, and saves back to the same path.
    """
    if not os.path.exists(file_path):
        print(f"Error: File not found at path: {file_path}")
        return

    print(f"Loading Excel file: {file_path} ...")
    df = pd.read_excel(file_path)

    # Check if the project column exists (case-insensitive check)
    target_col = None
    for col in df.columns:
        if col.strip().lower() == project_col_name.lower():
            target_col = col
            break

    if not target_col:
        print(
            f"Error: Column '{project_col_name}' not found in Excel sheet. Available columns: {list(df.columns)}"
        )
        return

    print(f"Classifying project names using column '{target_col}' ...")

    # Apply classification row-by-row
    classifications = df[target_col].apply(
        lambda x: classify_text(x, TAXONOMY)
    )

    # Unpack classification results into separate lists
    main_headings = [c[0] for c in classifications]
    subheadings = [c[1] for c in classifications]
    keywords = [c[2] for c in classifications]

    # Add or update the new columns in the DataFrame
    df["Main Heading"] = main_headings
    df["Subheading"] = subheadings
    df["Matched Keyword"] = keywords

    # Save back to the same Excel file
    print(f"Saving updated DataFrame back to: {file_path} ...")
    df.to_excel(file_path, index=False)
    print("Process completed successfully!")


if __name__ == "__main__":
    # Your file path

   

# 1. Path Configuration
    excel_path = os.getenv("BRONZE") + r"\RAW_MERGED.xlsx"
    

    # Pass the path and your project column name if it differs from 'Project Name'
    update_excel_file(excel_path, project_col_name="Project Name")

Loading Excel file: F:\Chimney Work\Marketing\Parivesh Work\Data Architecture\Bronze\RAW_MERGED.xlsx ...
Classifying project names using column 'Project Name' ...
Saving updated DataFrame back to: F:\Chimney Work\Marketing\Parivesh Work\Data Architecture\Bronze\RAW_MERGED.xlsx ...
Process completed successfully!
